# Exp20 — Preflight for round-two tasks: word order, translation (scaled), ZH classifier (fixed), AR numeral polarity

Same gate as Exp18/19 (accuracy >= 0.75, Wilson 95% lower bound > 0.5, >= 90% of items scorable). Three changes from Exp19:

1. **Candidates are full noun phrases / full sentences, not bare words.** In Exp19 the bare candidate let a shorter tokenization win by length alone (this would have hit Arabic numerals: ثلاث vs ثلاثة). Now both candidates contain the same words, so length differences come only from the manipulated feature.
2. **Word order** is scored as sentence log-prob of the grammatical vs a permuted order, after a newline prefix, in three sub-phenomena: adjective-noun, clause order (verb position), adposition placement.
3. **Translation** pulls ~150 items per language from the MUSE bilingual dictionaries instead of hand-writing them. **Download is untested**: if the server cannot reach dl.fbaipublicfiles.com, put `en-fr.txt`, `en-es.txt`, `en-de.txt`, `en-zh.txt`, `en-ar.txt`, `en-hi.txt` in `./muse_dicts/` and re-run.

**Predicted epsilon for word order is written below, before any run**: the number of basic order features (adjective-noun, clause order, adposition) that mismatch English. Verify the classifications against WALS (features 81A, 85A, 87A) before citing them.

Everything hand-written needs a native-speaker read. The ZH clause-order items are the weakest: Chinese allows object preposing, so the permuted order is marked rather than clearly ungrammatical.

In [1]:
# CELL 1 — config + data (edit MODELS / GPU_ID only)
import os
GPU_ID = 2
os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(GPU_ID))
import json, math, random, difflib, urllib.request
from pathlib import Path
OUT = Path("preflight_results"); OUT.mkdir(exist_ok=True)
MUSE_DIR = Path("muse_dicts")
MUSE_URL = "https://dl.fbaipublicfiles.com/arrival/dictionaries/en-{l}.txt"

MODELS = {"bloom-1b7": "bigscience/bloom-1b7", "bloom-7b1": "bigscience/bloom-7b1"}
def prepend_bos(key): return "gemma" in key
ACC_GATE, COVER_GATE, SEED = 0.75, 0.90, 0
N_TRANSLATION, N_DEMOS = 150, 5
LANGNAME = {"fr": "French", "es": "Spanish", "de": "German", "zh": "Chinese", "ar": "Arabic", "hi": "Hindi"}

# ---- predicted epsilon for word order, written BEFORE running: # of {adj-noun, clause order, adposition} features mismatching EN
PRED_EPS_ORDER = {"en": 0, "de": 0, "fr": 1, "es": 1, "zh": 1, "ar": 2, "hi": 2}
PRED_WHY = {"de": "Adj-N, SVO main clause, prepositions (verb-final subordinate clauses not tested)", "fr": "N-Adj",
            "es": "N-Adj", "zh": "circumposition (zai...li)", "ar": "N-Adj + VSO", "hi": "SOV + postpositions"}

# ---- WORD ORDER (full sentences; good order first, permuted order second)
ORDER_CLAUSE = {
 "en": [("The boy read the book.","The boy the book read."),("She bought a car.","She a car bought."),("The dog chased the cat.","The dog the cat chased."),
        ("We saw the movie.","We the movie saw."),("The teacher opened the door.","The teacher the door opened."),("They built a house.","They a house built.")],
 "fr": [("Le garçon lit le livre.","Le garçon le livre lit."),("Elle achète une voiture.","Elle une voiture achète."),("Le chien poursuit le chat.","Le chien le chat poursuit."),
        ("Nous voyons le film.","Nous le film voyons."),("Le professeur ouvre la porte.","Le professeur la porte ouvre."),("Ils construisent une maison.","Ils une maison construisent.")],
 "es": [("El niño lee el libro.","El niño el libro lee."),("Ella compra un coche.","Ella un coche compra."),("El perro persigue al gato.","El perro al gato persigue."),
        ("Nosotros vemos la película.","Nosotros la película vemos."),("El profesor abre la puerta.","El profesor la puerta abre."),("Ellos construyen una casa.","Ellos una casa construyen.")],
 "de": [("Der Junge liest das Buch.","Der Junge das Buch liest."),("Sie kauft ein Auto.","Sie ein Auto kauft."),("Der Hund jagt die Katze.","Der Hund die Katze jagt."),
        ("Wir sehen den Film.","Wir den Film sehen."),("Der Lehrer öffnet die Tür.","Der Lehrer die Tür öffnet."),("Sie bauen ein Haus.","Sie ein Haus bauen.")],
 "zh": [("男孩读那本书。","男孩那本书读。"),("她买一辆车。","她一辆车买。"),("狗追猫。","狗猫追。"),("我们看电影。","我们电影看。"),("老师打开门。","老师门打开。"),("他们盖房子。","他们房子盖。")],
 "ar": [("قرأ الولد الكتاب.","الولد الكتاب قرأ."),("فتح المعلم الباب.","المعلم الباب فتح."),("كتب الطالب الرسالة.","الطالب الرسالة كتب."),
        ("شاهد الأطفال الفيلم.","الأطفال الفيلم شاهد."),("بنى الرجل بيتا.","الرجل بيتا بنى."),("أكل الولد التفاحة.","الولد التفاحة أكل.")],
 "hi": [("लड़के ने किताब पढ़ी।","लड़के ने पढ़ी किताब।"),("उसने गाड़ी खरीदी।","उसने खरीदी गाड़ी।"),("कुत्ते ने बिल्ली का पीछा किया।","कुत्ते ने किया बिल्ली का पीछा।"),
        ("हमने फ़िल्म देखी।","हमने देखी फ़िल्म।"),("शिक्षक ने दरवाज़ा खोला।","शिक्षक ने खोला दरवाज़ा।"),("उन्होंने घर बनाया।","उन्होंने बनाया घर।")],
}
ORDER_ADP = {
 "en": [("The cat sleeps in the house.","The cat sleeps the house in."),("She walked to the school.","She walked the school to."),("He works with his father.","He works his father with."),
        ("They live near the river.","They live the river near."),("The book is on the table.","The book is the table on."),("We arrived after the storm.","We arrived the storm after.")],
 "fr": [("Le chat dort dans la maison.","Le chat dort la maison dans."),("Elle est allée à l'école.","Elle est allée l'école à."),("Il travaille avec son père.","Il travaille son père avec."),
        ("Ils habitent près de la rivière.","Ils habitent la rivière près de."),("Le livre est sur la table.","Le livre est la table sur."),("Nous sommes arrivés après la tempête.","Nous sommes arrivés la tempête après.")],
 "es": [("El gato duerme en la casa.","El gato duerme la casa en."),("Ella fue a la escuela.","Ella fue la escuela a."),("Él trabaja con su padre.","Él trabaja su padre con."),
        ("Viven cerca del río.","Viven del río cerca."),("El libro está sobre la mesa.","El libro está la mesa sobre."),("Llegamos después de la tormenta.","Llegamos la tormenta después de.")],
 "de": [("Die Katze schläft im Haus.","Die Katze schläft Haus im."),("Sie ging zur Schule.","Sie ging Schule zur."),("Er arbeitet mit seinem Vater.","Er arbeitet seinem Vater mit."),
        ("Sie wohnen am Fluss.","Sie wohnen Fluss am."),("Das Buch liegt auf dem Tisch.","Das Buch liegt dem Tisch auf."),("Wir sprachen über das Problem.","Wir sprachen das Problem über.")],
 "zh": [("猫在房子里睡觉。","猫房子里在睡觉。"),("书在桌子上。","书桌子上在。"),("他们住在河边。","他们住河边在。"),("孩子在学校里学习。","孩子学校里在学习。"),("她坐在车上。","她坐车上在。"),("他躺在床上。","他躺床上在。")],
 "ar": [("تنام القطة في البيت.","تنام القطة البيت في."),("ذهبت إلى المدرسة.","ذهبت المدرسة إلى."),("يعمل مع والده.","يعمل والده مع."),
        ("يسكنون قرب النهر.","يسكنون النهر قرب."),("الكتاب على الطاولة.","الكتاب الطاولة على."),("وصلنا بعد العاصفة.","وصلنا العاصفة بعد.")],
 "hi": [("बिल्ली घर में सोती है।","बिल्ली में घर सोती है।"),("किताब मेज़ पर है।","किताब पर मेज़ है।"),("लड़का स्कूल में पढ़ता है।","लड़का में स्कूल पढ़ता है।"),
        ("वे नदी के पास रहते हैं।","वे के पास नदी रहते हैं।"),("हम तूफ़ान के बाद पहुँचे।","हम बाद के तूफ़ान पहुँचे।"),("वह स्कूल तक गया।","वह तक स्कूल गया।")],
}
ADJ_PRE = {"en": True, "de": True, "zh": True, "hi": True, "fr": False, "es": False, "ar": False}
ADJ_CARRIER = {"en": ("She bought", "."), "fr": ("Elle a acheté", "."), "es": ("Ella compró", "."), "de": ("Sie kaufte", "."),
               "zh": ("她买了", "。"), "ar": ("اشترت", "."), "hi": ("उसने", " खरीदी।")}
ADJ_NP = {   # (determiner/classifier, adjective, noun)
 "en": [("a","red","car"),("a","green","book"),("a","black","hat"),("a","white","shirt"),("a","blue","bag"),("a","round","table")],
 "fr": [("une","rouge","voiture"),("un","vert","livre"),("un","noir","chapeau"),("une","blanche","chemise"),("un","bleu","sac"),("une","ronde","table")],
 "es": [("un","rojo","coche"),("un","verde","libro"),("un","negro","sombrero"),("una","blanca","camisa"),("un","azul","bolso"),("una","redonda","mesa")],
 "de": [("ein","rotes","Auto"),("ein","grünes","Buch"),("einen","schwarzen","Hut"),("ein","weißes","Hemd"),("eine","blaue","Tasche"),("einen","runden","Tisch")],
 "zh": [("一辆","红色的","车"),("一本","绿色的","书"),("一顶","黑色的","帽子"),("一件","白色的","衬衫"),("一个","蓝色的","包"),("一张","圆形的","桌子")],
 "ar": [("","حمراء","سيارة"),("","أخضر","كتابا"),("","سوداء","قبعة"),("","أبيض","قميصا"),("","زرقاء","حقيبة"),("","مستديرة","طاولة")],
 "hi": [("एक","लाल","गाड़ी"),("एक","हरी","किताब"),("एक","काली","टोपी"),("एक","सफ़ेद","कमीज़"),("एक","नीली","कुर्सी"),("एक","गोल","मेज़")],
}

# ---- ZH CLASSIFIER (candidate = numeral + classifier + noun; foil swaps in another noun's classifier)
CLF = [("书","本"),("桌子","张"),("狗","只"),("车","辆"),("花","朵"),("鱼","条"),("树","棵"),("房子","座")]
CLF_FMT = "昨天我在商店里看到了一些{n}。我买了"

# ---- AR NUMERAL POLARITY (3-10 take the opposite gender of the counted noun). cue=True: sound feminine plural (-at)
NUM_AR = [("كتب","m",False),("أقلام","m",False),("أبواب","m",False),("مفاتيح","m",False),
          ("سيارات","f",True),("طاولات","f",True),("مدارس","f",False),("حقائب","f",False)]
NUMERALS_AR = [("ثلاثة","ثلاث"),("أربعة","أربع"),("خمسة","خمس"),("ستة","ست")]    # (with ta marbuta = masculine noun, without = feminine noun)

TASK_LANGS = {"word_order": ["en","fr","es","de","zh","ar","hi"], "classifier": ["zh"], "numeral_polarity": ["ar"],
              "translation": ["fr","es","de","zh","ar","hi"]}
print("config ok")

config ok


In [2]:
# CELL 2 — tokenizer helpers + item builders
def enc(tok, text): return tok(text, add_special_tokens=False)["input_ids"]
def new_tokens(tok, prefix, cont):
    a, b = enc(tok, prefix), enc(tok, prefix + cont)
    return b[len(a):] if b[:len(a)] == a else None
def mk(task, lang, prefix, correct, foil, sep, **meta): return dict(task=task, lang=lang, prefix=prefix, correct=correct, foil=foil, sep=sep, meta=meta)

NL = "\n"
def build_word_order(lang):
    items = [mk("word_order", lang, NL, g, b, "", sub="clause") for g, b in ORDER_CLAUSE[lang]]
    items += [mk("word_order", lang, NL, g, b, "", sub="adposition") for g, b in ORDER_ADP[lang]]
    left, right = ADJ_CARRIER[lang]; join = "" if lang == "zh" else " "
    for art, adj, noun in ADJ_NP[lang]:
        pre = [x for x in (left, art, adj, noun) if x]; post = [x for x in (left, art, noun, adj) if x]
        good, bad = (pre, post) if ADJ_PRE[lang] else (post, pre)
        items.append(mk("word_order", lang, NL, join.join(good) + right, join.join(bad) + right, "", sub="adj_noun"))
    return items

def build_classifier(lang="zh"):
    cl, items = [c for _, c in CLF], []
    for i, (n, c) in enumerate(CLF):
        for j in (1, 2):
            items.append(mk("classifier", lang, CLF_FMT.format(n=n), f"三{c}{n}", f"三{cl[(i + j) % len(cl)]}{n}", ""))
    return items

def build_numeral_polarity(lang="ar"):
    items = []
    for i, (n, g, cue) in enumerate(NUM_AR):
        for j in (0, 1):
            m, f = NUMERALS_AR[(i + j) % len(NUMERALS_AR)]
            c, x = (m, f) if g == "m" else (f, m)
            items.append(mk("numeral_polarity", lang, f"في المتجر {n}. اشتريت", f"{c} {n}", f"{x} {n}", " ", gender=g, cue=cue))
    return items

def load_muse(l):
    p = MUSE_DIR / f"en-{l}.txt"
    if not p.exists():
        try: MUSE_DIR.mkdir(exist_ok=True); urllib.request.urlretrieve(MUSE_URL.format(l=l), p)
        except Exception as e: print("MUSE download failed for", l, "->", e, "| place en-%s.txt in %s/" % (l, MUSE_DIR)); return None
    return [tuple(x.split()) for x in p.read_text(encoding="utf-8").splitlines() if len(x.split()) == 2]

def clean_pairs(l, pairs):
    seen_s, seen_t, out = set(), set(), []
    for s, t in pairs:
        if not (s.isascii() and s.isalpha() and s.islower() and len(s) >= 3): continue
        if s in seen_s or t in seen_t or t.casefold() == s: continue
        if l in ("fr", "es", "de") and (not t.isalpha() or difflib.SequenceMatcher(None, s, t.casefold()).ratio() >= 0.8): continue   # drop cognates / copy shortcuts
        seen_s.add(s); seen_t.add(t); out.append((s, t))
    return out

def build_translation(lang):
    pairs = load_muse(lang)
    if not pairs: return []
    pairs = clean_pairs(lang, pairs); rng = random.Random(SEED); rng.shuffle(pairs)
    demos, test = pairs[:20], pairs[20:20 + N_TRANSLATION]
    tg, items = LANGNAME[lang], []
    for s, t in test:
        ds = rng.sample(demos, N_DEMOS)
        prefix = "".join(f"English: {a}\n{tg}: {b}\n\n" for a, b in ds) + f"English: {s}\n{tg}:"
        pool = [y for _, y in test if y != t and abs(len(y) - len(t)) <= 2] or [y for _, y in test if y != t]
        items.append(mk("translation", lang, prefix, t, rng.choice(pool), " "))
    return items

BUILD = {"word_order": build_word_order, "classifier": build_classifier, "numeral_polarity": build_numeral_polarity, "translation": build_translation}
ITEMS = {}
for t, ls in TASK_LANGS.items():
    for l in ls:
        its = BUILD[t](l)
        if its: ITEMS[(t, l)] = its
print({f"{t}|{l}": len(v) for (t, l), v in ITEMS.items()})

{'word_order|en': 18, 'word_order|fr': 18, 'word_order|es': 18, 'word_order|de': 18, 'word_order|zh': 18, 'word_order|ar': 18, 'word_order|hi': 18, 'classifier|zh': 16, 'numeral_polarity|ar': 16, 'translation|fr': 150, 'translation|es': 150, 'translation|de': 150, 'translation|zh': 150, 'translation|ar': 150, 'translation|hi': 150}


In [3]:
# CELL 3 — coverage + candidate collisions (CPU)
import pandas as pd
from transformers import AutoTokenizer
TOK, COVER = {}, []
for key, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path); TOK[key] = tok
    for (t, l), items in ITEMS.items():
        ok = coll = difflen = 0
        for it in items:
            c, f = new_tokens(tok, it["prefix"], it["sep"] + it["correct"]), new_tokens(tok, it["prefix"], it["sep"] + it["foil"])
            if c and f: ok += 1; coll += int(c[0] == f[0]); difflen += int(len(c) != len(f))
        COVER.append(dict(model=key, task=t, lang=l, n=len(items), scorable=ok / len(items), first_token_collision=coll / max(1, ok), length_differs=difflen / max(1, ok)))
df_cover = pd.DataFrame(COVER); print(df_cover.round(2).to_string(index=False))

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    model             task lang   n  scorable  first_token_collision  length_differs
bloom-1b7       word_order   en  18       1.0                   1.00            0.00
bloom-1b7       word_order   fr  18       1.0                   1.00            0.00
bloom-1b7       word_order   es  18       1.0                   1.00            0.00
bloom-1b7       word_order   de  18       1.0                   1.00            0.00
bloom-1b7       word_order   zh  18       1.0                   1.00            0.33
bloom-1b7       word_order   ar  18       1.0                   0.67            0.33
bloom-1b7       word_order   hi  18       1.0                   1.00            0.00
bloom-1b7       classifier   zh  16       1.0                   0.75            0.38
bloom-1b7 numeral_polarity   ar  16       1.0                   0.00            0.00
bloom-1b7      translation   fr 150       1.0                   0.00            0.60
bloom-1b7      translation   es 150       1.0                   0

In [4]:
# CELL 4 — score (full-sequence log-prob of each candidate)
import torch, gc
from transformers import AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def cand_lp(model, ids_prefix, cand_ids):
    ids = torch.tensor([ids_prefix + cand_ids], device=dev)
    lp = torch.log_softmax(model(ids).logits[0].float(), dim=-1)
    base = len(ids_prefix) - 1
    return [lp[base + j, t].item() for j, t in enumerate(cand_ids)]

RES = []
for key, path in MODELS.items():
    try: model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16).to(dev).eval()
    except Exception as e: print("LOAD FAILED", key, e); continue
    tok = TOK[key]; bos = [tok.bos_token_id] if prepend_bos(key) else []
    for (t, l), items in ITEMS.items():
        for it in items:
            c, f = new_tokens(tok, it["prefix"], it["sep"] + it["correct"]), new_tokens(tok, it["prefix"], it["sep"] + it["foil"])
            if not (c and f): RES.append(dict(model=key, task=t, lang=l, scored=False, **it["meta"])); continue
            p = bos + enc(tok, it["prefix"]); lc, lf = cand_lp(model, p, c), cand_lp(model, p, f)
            RES.append(dict(model=key, task=t, lang=l, scored=True, margin_sum=sum(lc) - sum(lf), margin_mean=sum(lc) / len(lc) - sum(lf) / len(lf), **it["meta"]))
    del model; gc.collect(); torch.cuda.empty_cache(); print("done", key)
df_res = pd.DataFrame(RES)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 2375.18it/s]


done bloom-1b7


Loading weights: 100%|██████████| 365/365 [00:00<00:00, 962.61it/s] 


done bloom-7b1


In [5]:
# CELL 5 — verdict
def wilson_lo(k, n, z=1.96):
    if n == 0: return float("nan")
    p = k / n; d = 1 + z * z / n
    return ((p + z * z / (2 * n)) - z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d

SPLIT = {"word_order": "sub", "numeral_polarity": "cue"}
rows = []
for (key, t, l), g in df_res.groupby(["model", "task", "lang"]):
    s = g[g.scored]; n_all, n = len(g), len(s); k = int((s.margin_sum > 0).sum()) if n else 0
    r = dict(model=key, task=t, lang=l, n=n_all, scorable=round(n / n_all, 2), acc=round(k / n, 3) if n else float("nan"), ci_lo=round(wilson_lo(k, n), 2),
             acc_mean_norm=round(float((s.margin_mean > 0).mean()), 3) if n else float("nan"), mean_margin=round(float(s.margin_sum.mean()), 2) if n else float("nan"))
    if t in SPLIT and n:
        for v, gg in s.groupby(SPLIT[t]): r[f"acc[{SPLIT[t]}={v}]"] = round(float((gg.margin_sum > 0).mean()), 3)
    if t == "word_order": r["pred_eps"] = PRED_EPS_ORDER[l]
    r["PASS"] = bool(n and n / n_all >= COVER_GATE and k / n >= ACC_GATE and wilson_lo(k, n) > 0.5)
    rows.append(r)
df_verdict = pd.DataFrame(rows).sort_values(["task", "lang", "model"])
pd.set_option("display.width", 250); print(df_verdict.to_string(index=False))
print("\nPASS matrix (all models must pass):")
print(df_verdict.pivot_table(index="task", columns="lang", values="PASS", aggfunc="all").to_string())
df_verdict.to_csv(OUT / "verdict_round2.csv", index=False); df_cover.to_csv(OUT / "coverage_round2.csv", index=False); df_res.to_csv(OUT / "item_level_round2.csv", index=False)

    model             task lang   n  scorable   acc  ci_lo  acc_mean_norm  mean_margin  PASS  acc[cue=False]  acc[cue=True]  acc[sub=adj_noun]  acc[sub=adposition]  acc[sub=clause]  pred_eps
bloom-1b7       classifier   zh  16       1.0 0.938   0.72          0.938         6.36  True             NaN            NaN                NaN                  NaN              NaN       NaN
bloom-7b1       classifier   zh  16       1.0 1.000   0.81          1.000         7.17  True             NaN            NaN                NaN                  NaN              NaN       NaN
bloom-1b7 numeral_polarity   ar  16       1.0 0.938   0.72          0.938         1.54  True           0.917            1.0                NaN                  NaN              NaN       NaN
bloom-7b1 numeral_polarity   ar  16       1.0 0.938   0.72          0.938         1.17  True           0.917            1.0                NaN                  NaN              NaN       NaN
bloom-1b7      translation   ar 150       1.0